# Advanced EDA: Superstore Sales Analysis
This notebook explores correlation, outliers, and hypothesis testing on Superstore sales data




In [3]:
import pandas as pd
from scipy import stats

df = pd.read_csv('/content/superstore.csv', encoding='latin-1')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

## Correlation Findings

We looked at how Sales, Quantity, Discount, and Profit relate to each other.

- **Discount and Profit** show a weak-to-moderate negative relationship (-0.22). As expected, higher discounts tend to reduce profit, but the relationship isn't very strong, which suggests profit is also affected by other factors (like the base price of the product or how many units were sold), not discount alone.

- **Sales and Profit** show a moderate positive relationship (0.48). Larger sales transactions tend to generate more profit, though not perfectly some high-sales orders may still have low profit if they came with heavy discounts.

- **Quantity and Sales** show only a weak relationship (0.20). This makes sense because Sales depends on both quantity AND price a single expensive item (like furniture) can generate high Sales with low Quantity, while several cheap items (like office supplies) can do the opposite.

In [5]:
df.corr(numeric_only=True)

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
Row ID,1.000000,0.009671,-0.001359,-0.004016,0.013480,0.012497
Postal Code,0.009671,1.000000,-0.023854,0.012761,0.058443,-0.029961
Sales,-0.001359,-0.023854,1.000000,0.200795,-0.028190,0.479064
Quantity,-0.004016,0.012761,0.200795,1.000000,0.008623,0.066253
Discount,0.013480,0.058443,-0.028190,0.008623,1.000000,-0.219487
Profit,0.012497,-0.029961,0.479064,0.066253,-0.219487,1.000000


## Outlier Detection: IQR vs Z-Score

We tested two methods to detect unusual Profit values: IQR and Z-score.

The IQR method flagged 1,881 transactions (about 19% of the dataset) as outliers this is too large a proportion to reasonably call "outliers," and treating this much data as unusual would mean ignoring a big chunk of otherwise valid business activity.

The Z-score method flagged only 107 transactions (about 1% of the dataset) a much more reasonable and usable result.

This difference likely comes from the shape of the Profit data itself: a superstore sells everything from cheap pens to expensive furniture, so Profit naturally varies a lot across transactions. IQR's method (based on the tight middle 50% of the data) ends up too strict for this kind of varied, real-world data, while Z-score's approach (based on overall average and spread) handled this variation more sensibly.

In [6]:
Q1 = df['Profit'].quantile(0.25)
Q3 = df['Profit'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Profit'] < lower_bound) | (df['Profit'] > upper_bound)]

In [7]:
outliers.shape

(1881, 21)

In [8]:
print(lower_bound, upper_bound)

-39.724125 70.816875


In [9]:
profit_mean = df['Profit'].mean()
print(profit_mean)

28.65689630778467


In [10]:
profit_std = df['Profit'].std()
print(profit_std)

234.26010769095757


In [11]:
df['z_score'] = (df['Profit'] - profit_mean) / profit_std

In [12]:
outliers_z = df[df['z_score'].abs() > 3]
outliers_z.shape

(107, 22)

In [13]:
df['z_score'] = (df['Profit'] - df['Profit'].mean()) / df['Profit'].std()
outliers_z = df[df['z_score'].abs() > 3]
outliers_z.shape

(107, 22)

## Hypothesis Testing: Region and Shipping Mode

We ran two statistical tests to check whether certain business groupings show a real difference in average Profit, or whether any observed gap could just be random variation.

**East vs West Region**
The p-value came out to 0.756, which is well above the 0.05 threshold. This means we don't have enough evidence to conclude that East and West regions have genuinely different average profits any gap between them is small enough to plausibly be random noise rather than a real business difference.

**First Class vs Standard Class Shipping**
First Class orders averaged 31 dollar in profit, compared to 27.49 dollar for Standard Class a $4.35 gap. However, the p-value for this comparison was 0.546, also well above 0.05. So despite the visible gap in averages, we don't have enough evidence to call this a reliable, real difference individual order profits vary widely regardless of shipping method, which can easily produce a gap this size by chance alone.

**Overall takeaway:** neither region nor shipping mode showed a statistically significant effect on profit in this dataset. This is a useful finding in itself it suggests profit is likely driven more by factors like discount level and product category than by where an order ships to or how fast it ships.

In [14]:
east_profit = df[df['Region'] == 'East']['Profit']
west_profit = df[df['Region'] == 'West']['Profit']

In [15]:
t_stat, p_value = stats.ttest_ind(east_profit, west_profit)
print(p_value)

0.7560510387066459


In [16]:
df.groupby('Region')['Profit'].mean()

,Profit
Region,
Central,17.092709
East,32.135808
South,28.857673
West,33.849032


In [17]:
group1 = df[df['Ship Mode'] == 'First Class']['Profit']
group2 = df[df['Ship Mode'] == 'Standard Class']['Profit']

t_stat, p_value = stats.ttest_ind(group1, group2)
print(p_value)

0.5459487157344598


In [18]:
df.groupby('Ship Mode')['Profit'].mean()

,Profit
Ship Mode,
First Class,31.839948
Same Day,29.266591
Second Class,29.535545
Standard Class,27.494770
